
# 🎨 ClawSouls — Avatar API Server (FastAPI + Cloudflared)

Sobe um servidor **FastAPI** no Colab que gera avatares por requisição HTTP.

**Fluxo:**
1. Rode este notebook (com GPU)
2. Copie a URL pública do Cloudflared
3. Envie POST `/generate` com os atributos da soul
4. Receba a imagem gerada em base64

---



## Pré-requisitos

- `Runtime > Change runtime type > T4 GPU`
- Não precisa do repositório clonado (tudo é self-contained)


In [ ]:

# ═══════════════════════════════════════════════════════════
# TOGGLE DE MODELO
# ═══════════════════════════════════════════════════════════

MODELO = "sdxl"          # ← "sdxl" ou "turbo"
SECRET_TOKEN = "cs-secret-2026"  # ← Troque para algo seguro!
TOTAL_STEPS = 15          # Override global de steps
TOTAL_GUIDANCE = 7.5      # Override global de guidance

CATALOGO = {
    "turbo": {
        "model_id": "Tongyi-MAI/Z-Image-Turbo",
        "width": 512, "height": 768, "variant": "fp32",
        "default_steps": 4, "default_guidance": 1.0,
    },
    "sdxl": {
        "model_id": "stabilityai/stable-diffusion-xl-base-1.0",
        "width": 512, "height": 768, "variant": "fp16",
        "default_steps": 25, "default_guidance": 7.5,
    },
}

assert MODELO in CATALOGO, f"Use: {list(CATALOGO.keys())}"
cfg = CATALOGO[MODELO]
MODEL_ID = cfg["model_id"]

print(f"🔧 Modelo: {MODELO} ({MODEL_ID})")
print(f"   Steps: {TOTAL_STEPS}, Guidance: {TOTAL_GUIDANCE}, Token: {SECRET_TOKEN[:4]}...")


In [ ]:

# Escreve o servidor num arquivo .py (servidor thin — só recebe prompt e gera)
# O prompt é montado do lado do agente; o servidor é burro e direto.

import textwrap

server_code = textwrap.dedent('''

from urllib.parse import parse_qs
import io, base64, time, os, re
from datetime import datetime
from typing import Optional

from pydantic import BaseModel
from fastapi import FastAPI, HTTPException, Request
import torch
from diffusers import AutoPipelineForText2Image

# ── Config (injetadas pelo notebook via env vars) ──────────
MODELO = os.environ.get('CLAWSOULS_MODELO', 'sdxl')
SECRET_TOKEN = os.environ.get('CLAWSOULS_SECRET', 'cs-secret-2026')
TOTAL_STEPS = int(os.environ.get('CLAWSOULS_STEPS', '15'))
TOTAL_GUIDANCE = float(os.environ.get('CLAWSOULS_GUIDANCE', '7.5'))
MODEL_ID = os.environ.get('CLAWSOULS_MODEL_ID', 'stabilityai/stable-diffusion-xl-base-1.0')
MODEL_WIDTH = int(os.environ.get('CLAWSOULS_WIDTH', '512'))
MODEL_HEIGHT = int(os.environ.get('CLAWSOULS_HEIGHT', '768'))
MODEL_VARIANT = os.environ.get('CLAWSOULS_VARIANT', 'fp16')

DEFAULT_NEGATIVE = (
    'blurry, low quality, deformed, ugly, duplicate, disfigured, '
    'bad anatomy, bad proportions, extra limbs, mutated hands, '
    'text, watermark, signature, logo, '
    'photorealistic, 3d render, '
    'nude, NSFW, gore'
)

# ── FastAPI App ────────────────────────────────────────
class GenerateRequest(BaseModel):
    name: Optional[str] = 'unnamed'
    custom_prompt: str
    steps: Optional[int] = None
    guidance: Optional[float] = None
    custom_negative_prompt: Optional[str] = None

app = FastAPI(title='ClawSouls Avatar API', version='2.0')

def _check_token(request: Request) -> bool:
    auth = request.headers.get('Authorization', '')
    if auth == 'Bearer ' + SECRET_TOKEN:
        return True
    qs = parse_qs(request.url.query)
    if qs.get('token', [''])[0] == SECRET_TOKEN:
        return True
    if request.headers.get('X-Token') == SECRET_TOKEN:
        return True
    return False

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODELO, 'model_id': MODEL_ID}

@app.get('/models')
def list_models():
    return {'available': {}, 'current': MODELO}

@app.post('/generate')
async def generate(request: Request, req: GenerateRequest):
    if not _check_token(request):
        raise HTTPException(status_code=401, detail='Unauthorized')
    prompt = req.custom_prompt
    negative = req.custom_negative_prompt or DEFAULT_NEGATIVE
    steps = req.steps if req.steps else TOTAL_STEPS
    guidance = req.guidance if req.guidance else TOTAL_GUIDANCE
    print(f\"[{datetime.now().strftime('%H:%M:%S')}] Generating: {req.name} ({steps} steps)\")
    start = time.time()
    generator = torch.Generator(device='cuda').manual_seed(int(time.time() * 1000) % (2**32))
    image = pipe(
        prompt=prompt, negative_prompt=negative,
        num_inference_steps=steps, guidance_scale=guidance,
        width=MODEL_WIDTH, height=MODEL_HEIGHT, generator=generator,
    ).images[0]
    elapsed = time.time() - start
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode('utf-8')
    safe_name = ''.join(c if c.isalnum() or c in '._-' else '_' for c in req.name.lower().strip())
    print(f\"   ✅ {req.name} → {elapsed:.1f}s\")
    return {
        'name': req.name, 'slug': safe_name,
        'prompt': prompt, 'negative_prompt': negative,
        'seed': int(generator.initial_seed()),
        'steps': steps, 'guidance': guidance, 'model': MODELO,
        'elapsed_s': round(elapsed, 1), 'image_base64': img_b64,
    }

# ── Lifespan (model loading) ─────────────────────────────
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app):
    global pipe

    hf_token = os.environ.get('HF_TOKEN', '')
    if not hf_token:
        token_path = os.path.expanduser('~/.cache/huggingface/token')
        if os.path.exists(token_path):
            with open(token_path) as f:
                hf_token = f.read().strip()

    print(f\"Loading model: {MODEL_ID}\")
    print(f\"  variant={MODEL_VARIANT}, token={'set' if hf_token else 'NONE'}\")

    attempt_variants = [MODEL_VARIANT, None]
    pipe = None

    for attempt_variant in attempt_variants:
        try:
            print(f\"  Trying variant={attempt_variant}...\")
            pipe = AutoPipelineForText2Image.from_pretrained(
                MODEL_ID,
                torch_dtype=torch.float16 if MODEL_VARIANT == 'fp16' else torch.float32,
                variant=attempt_variant,
                use_safetensors=True,
                token=hf_token if hf_token else None,
            )
            pipe.enable_attention_slicing()
            if hasattr(pipe, 'enable_vae_tiling'):
                pipe.enable_vae_tiling()
            pipe.to('cuda')
            if hasattr(pipe, 'vae') and MODEL_VARIANT == 'fp16':
                pipe.vae.to('cuda', torch.float16)
            print(f'  ✅ Modelo carregado! (variant={attempt_variant})')
            break
        except Exception as e:
            print(f'  ⚠️  Falhou com variant={attempt_variant}: {e}')
            if attempt_variant is None:
                print('  ❌ Todas as tentativas falharam!')
                raise
            continue

    if pipe is None:
        raise RuntimeError('Não foi possível carregar o modelo!')

    yield
    del pipe
    torch.cuda.empty_cache()
    print('🛑 Servidor encerrado')

app.router.lifespan_context = lifespan

''')


In [ ]:

# Silenciar warnings de Flax
import warnings, os
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

!pip install -q fastapi uvicorn[standard] pydantic pillow
!pip install -q diffusers[torch] transformers accelerate torch torchvision safetensors huggingface_hub

# cloudflared não está no PyPI — baixa o binário direto do GitHub
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("✅ Dependências instaladas!")


In [ ]:

# Inicia uvicorn rodando server.py em background
uvicorn_proc = !nohup python3 -m uvicorn server:app --host 0.0.0.0 --port 8000 --log-level warning > /tmp/uvicorn.log 2>&1 &

# Healthcheck: espera o servidor responder antes de subir o tunnel
import time, urllib.request

print('⏳ Esperando servidor iniciar...')
server_ok = False
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        server_ok = True
        print('✅ Servidor ativo!')
        break
    except Exception as e:
        time.sleep(1)

if not server_ok:
    print('⚠️  Servidor não respondeu em 30s!')
    !cat /tmp/uvicorn.log


---

## Túnel Cloudflared


In [ ]:

import subprocess
import re

# Inicia cloudflared tunnel em background
cloudflared_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Espera a URL do tunnel aparecer nos logs
tunnel_url = None
print('⏳ Esperando URL do tunnel...')
for i in range(30):
    line = cloudflared_proc.stdout.readline().decode('utf-8', errors='replace')
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print()
    print('=' * 60)
    print('🌐 TÚNEL ATIVO!')
    print(f'📎 URL pública: {tunnel_url}')
    print()
    print('Endpoints:')
    print(f'  GET  {tunnel_url}/health')
    print(f'  GET  {tunnel_url}/models')
    print(f'  POST {tunnel_url}/generate')
    print('=' * 60)
    print()
    print('⚠️  TOKEN: ' + SECRET_TOKEN)
    print()
    print('📋 Copie esta URL e cole aqui no chat para eu usar!')
else:
    print('⚠️  Não foi possível obter a URL do tunnel.')
    print('   Verifique: cloudflared_proc.wait()')


---

## Como usar

Cole a URL do tunnel aqui no chat. Eu monto o prompt e faço a requisição!

**Exemplo mínimo:**

```
POST /generate?token=cs-secret-2026
{
  "custom_prompt": "close-up portrait of a cyberpunk samurai, neon lights, detailed face, 4k"
}
```

**Exemplo completo:**

```
POST /generate?token=cs-secret-2026
{
  "name": "Kira",
  "custom_prompt": "close-up portrait, cyberpunk idol, neon glow, detailed face, masterpiece",
  "steps": 25,
  "guidance": 7.5
}
```
